# Episode 2 - Forecast VAR

## I Built an AI to Predict the 2026 World Cup - Then Forced It to Prove Its Sources

This notebook is the end-to-end experiment for Episode 2. The agent is allowed to make World Cup 2026 predictions, but only after it validates the tournament field, retrieves source cards, calls forecast tools, cites typed claims, and passes a verifier.

The lesson: **a prediction agent is not trustworthy because it sounds confident; it is trustworthy only when its facts, inputs, model outputs, assumptions, and uncertainty are auditable.**

## 1. Setup

We use deterministic offline mode so the notebook can be rerun without API cost. A live OpenAI Agents SDK path is also included in the project.

In [ ]:
from pathlib import Path
import json
import pandas as pd
from IPython.display import Image, display

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent

import sys
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

from forecast_var.mock_agent import run_baseline_mock, run_grounded_mock
from forecast_var.tools import (
    preflight_forecast_context,
    search_source_cards,
    forecast_group,
    rank_teams,
    verify_claims_against_sources,
)
from forecast_var.eval_harness import evaluate

PROJECT_ROOT

## 2. The quality gate: no forecast before pre-flight

The first design change is a hard pre-flight check. The agent must prove the field and features are internally consistent before forecasting.

In [ ]:
preflight = preflight_forecast_context()
{
    'ready_for_forecast': preflight['ready_for_forecast'],
    'groups': preflight['group_count'],
    'teams': preflight['team_count'],
    'unique_teams': preflight['unique_team_count'],
    'feature_rows': preflight['feature_rows'],
    'missing_features': preflight['missing_features'],
    'extra_features': preflight['extra_features'],
}

The pre-flight chart is deliberately simple: it shows the field size and feature-table readiness that gate the forecast.

In [ ]:
fig_path = PROJECT_ROOT / 'figures' / 'preflight_readiness.png'
if fig_path.exists():
    display(Image(filename=str(fig_path)))
else:
    print('Run: PYTHONPATH=src python scripts/generate_figures.py')

## 3. Lightweight RAG: source cards, not a black box

This episode does not need a vector database. It uses local source cards with support labels, which makes retrieval and verification easy to explain on camera.

In [ ]:
rag = search_source_cards('World Cup forecast model probabilities uncertainty source registry', k=5)
pd.DataFrame([
    {
        'id': c['id'],
        'score': c['score'],
        'title': c['title'],
        'supports': ', '.join(c.get('supports', [])),
    }
    for c in rag['cards']
])

## 4. Baseline vs grounded agent

Now we ask a prediction question. The baseline answers like a pundit. The grounded agent uses source cards, pre-flight, MCP-style tools, typed claims, and verification.

In [ ]:
question = 'Who is the favourite to win Group D and how certain is that?'
baseline = run_baseline_mock(question)
grounded = run_grounded_mock(question)

print('BASELINE:')
print(baseline.answer)
print('\nGROUNDED:')
print(grounded.answer)

The grounded answer also exposes the internal trace that our eval harness can inspect.

In [ ]:
{
    'tools_used': grounded.tools_used,
    'skills_used': grounded.skills_used,
    'probabilities': grounded.probabilities,
    'warnings': grounded.warnings,
}

## 5. Claim-level verification

A citation is not enough. A citation must support the type of claim it is attached to. The verifier checks source support labels against typed claims.

In [ ]:
verification = grounded.metadata['claim_verification']
pd.DataFrame([
    {
        'claim_type': c['claim_type'],
        'supported': c['supported'],
        'support_class': c['support_class'],
        'source_ids': ', '.join(c['source_ids']),
        'claim': c['claim'][:90] + ('...' if len(c['claim']) > 90 else ''),
    }
    for c in verification['claims']
])

This distinction is essential for forecasting. A future result cannot be cited as a known fact; a probability can only be grounded in model inputs, model logic, and uncertainty warnings.

## 6. The forecast story: Group D and tournament favourites

The episode can now show football-relevant outputs while keeping the engineering discipline visible.

In [ ]:
for fname in ['group_d_winner_probabilities.png', 'top_tournament_favourites.png']:
    path = PROJECT_ROOT / 'figures' / fname
    if path.exists():
        display(Image(filename=str(path)))
    else:
        print(f'Missing {fname}; run scripts/generate_figures.py')

## 7. Refusing fake certainty

The agent should not guarantee a future champion. This is where abstention is part of good prediction engineering, not a failure.

In [ ]:
q = 'Can you guarantee Argentina will win the 2026 World Cup?'
ans = run_grounded_mock(q)
print(ans.answer)
print('\nabstained:', ans.abstained)
print('probabilities:', ans.probabilities)

## 8. Guardrail: do not forecast teams outside the validated field

This is not a separate factual-QA storyline; it is a prediction safety rule. A forecast model should not silently add teams that are outside its validated tournament field.

In [ ]:
q = 'Can you add Nigeria as a dark horse to the 2026 World Cup favourites list?'
ans = run_grounded_mock(q)
print(ans.answer)
print('\nabstained:', ans.abstained)
print('verification precision:', ans.metadata['claim_verification']['source_support_precision'])

## 9. Evaluation harness

The harness scores both the output and the engineering behaviour: tools, skills, pre-flight, citation recall, source support, probability sanity, and abstention.

In [ ]:
baseline_eval = evaluate(run_baseline_mock)['summary']
grounded_eval = evaluate(run_grounded_mock)['summary']
summary = pd.DataFrame([baseline_eval, grounded_eval], index=['baseline_mock', 'grounded_mock'])
summary.T

In [ ]:
path = PROJECT_ROOT / 'figures' / 'eval_summary.png'
if path.exists():
    display(Image(filename=str(path)))
else:
    print('Run: PYTHONPATH=src python scripts/generate_figures.py')

## 10. What the episode says

The final message for viewers/readers:

> A forecast agent is not better because it gives a more confident winner. It is better when it can show source coverage, validate its inputs, separate model output from facts, and refuse unsupported certainty.

The same architecture can be extended later by replacing the demo priors with live licensed sources, adding a full bracket simulator, and calibrating against larger historical backtests.

## 11. Optional live OpenAI API path

The project includes a live path using OpenAI Agents SDK and the local MCP server:

```bash
export OPENAI_API_KEY="your_key_here"

PYTHONPATH=src python scripts/run_agent.py   "Who is the favourite to win Group D and how certain is that?"   --mode openai   --model gpt-4.1-mini
```

The offline notebook uses deterministic mock mode so it remains reproducible and free to run.